In [258]:
import pandas as pd
import numpy as np

In [259]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = df.drop('customerID', axis=1)
df.shape

(7043, 20)

In [260]:
df = df.dropna()

cols = df.select_dtypes(include='object').columns

mask = df[cols].apply(lambda col: col.astype(str).str.strip().eq('')).any(axis=1)

df = df[~mask]

C:\Users\User\AppData\Local\Temp\ipykernel_12096\3837072177.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cols = df.select_dtypes(include='object').columns


In [261]:
df.shape

(7032, 20)

In [262]:
df['TotalCharges'] = df['TotalCharges'].astype(float)
categorical_cols = df.select_dtypes(include='object').columns.tolist()
if 'Churn' in categorical_cols:
    categorical_cols.remove('Churn')


X = df.drop('Churn', axis=1).copy()
y = (df['Churn']).copy()
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)


C:\Users\User\AppData\Local\Temp\ipykernel_12096\245753736.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include='object').columns.tolist()


In [263]:
X.shape

(7032, 30)

In [264]:
X = X.astype(float)

In [265]:
X.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0.0,1.0,29.85,29.85,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,0.0,34.0,56.95,1889.50,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,0.0,2.0,53.85,108.15,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
3,0.0,45.0,42.30,1840.75,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,0.0,2.0,70.70,151.65,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


In [266]:
from sklearn.preprocessing import LabelEncoder

enc = LabelEncoder()
y = enc.fit_transform(y)



In [267]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

for col in X.columns:
    if X[col].max() > 3:
        st = StandardScaler()
        X[col] = st.fit_transform(X[[col]])

In [268]:
X.shape

(7032, 30)

In [269]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test =train_test_split(
    X,y,
    random_state=42,
    test_size=0.2,
    shuffle=True,
    stratify=y
)

In [270]:
X_train.shape

(5625, 30)

In [271]:
c = 0
for i in y_train:
    if i == 1:
        c += 1

c

1495

In [272]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, recall_score, precision_score

dummy = DummyClassifier()

dummy.fit(X_train, y_train)
p = dummy.predict(X_test)

f1_score(y_test, p)

0.0

In [273]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    class_weight='balanced',
    random_state=42
)
model.fit(X_train, y_train)
pred = model.predict(X_test)

pred = (np.array(pred) >= 0.4).astype(int)

print('logreg')
f1_score(y_test, pred)

logreg


0.6061224489795919

In [274]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train, y_train)

model1 = LogisticRegression(
    class_weight='balanced',
    random_state=42
)
model1.fit(X_smote, y_smote)
pred1 = model1.predict_proba(X_test)[:, 1]

pred1 = (np.array(pred1) >= 0.4).astype(int)

print('logreg_smote')
f1_score(y_test, pred1)

logreg_smote


0.6007462686567164

In [275]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42,
    class_weight='balanced',
    max_depth=8,
    min_samples_split=9,
    n_estimators=100,
    )


rf.fit(X_train, y_train)

pred2 = rf.predict_proba(X_test)[:, 1]

pred2 = (np.array(pred2) >= 0.5).astype(int)

print('fr')
f1_score(y_test, pred2)

fr


0.6187961985216474

In [276]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42,
    class_weight='balanced',
    max_depth=6,
    n_estimators=1000,
    min_samples_split=5,
    )

rf.fit(X_smote, y_smote)

pred3 = rf.predict(X_test)

print('fr_smote')
f1_score(y_test, pred3)

fr_smote


0.6310160427807486

In [277]:
from catboost import CatBoostClassifier


model3 = CatBoostClassifier(
    random_seed=42,
    learning_rate=0.01,
    loss_function='Logloss',
    eval_metric='F1',
    iterations=2000,
    depth=4,
    l2_leaf_reg=100
    )

model3.fit(X_train, y_train)
pred4 = model3.predict_proba(X_test)[:, 1]


pred4 = (np.array(pred4) >= 0.4).astype(int)
f1_score(y_test, pred4)

0:	learn: 0.5263158	total: 2.31ms	remaining: 4.62s
1:	learn: 0.5088908	total: 4.46ms	remaining: 4.46s
2:	learn: 0.5182452	total: 6.65ms	remaining: 4.42s
3:	learn: 0.5161826	total: 8.81ms	remaining: 4.4s
4:	learn: 0.5161826	total: 11.6ms	remaining: 4.62s
5:	learn: 0.4972091	total: 13.9ms	remaining: 4.6s
6:	learn: 0.5084459	total: 16ms	remaining: 4.56s
7:	learn: 0.4972091	total: 18.1ms	remaining: 4.51s
8:	learn: 0.4972091	total: 20.3ms	remaining: 4.49s
9:	learn: 0.5052989	total: 22.6ms	remaining: 4.49s
10:	learn: 0.5052989	total: 24.7ms	remaining: 4.46s
11:	learn: 0.5059322	total: 26.9ms	remaining: 4.46s
12:	learn: 0.5124106	total: 29.3ms	remaining: 4.48s
13:	learn: 0.5101351	total: 31.5ms	remaining: 4.47s
14:	learn: 0.5088608	total: 33.7ms	remaining: 4.46s
15:	learn: 0.5080169	total: 36ms	remaining: 4.47s
16:	learn: 0.5065650	total: 38.3ms	remaining: 4.47s
17:	learn: 0.5044586	total: 40.5ms	remaining: 4.46s
18:	learn: 0.5046809	total: 42.7ms	remaining: 4.45s
19:	learn: 0.5021386	total: 

0.6148055207026348

In [278]:
X.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0.0,-1.280248,-1.161694,-0.994194,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,0.0,0.064303,-0.260878,-0.173740,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,0.0,-1.239504,-0.363923,-0.959649,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
3,0.0,0.512486,-0.747850,-0.195248,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,0.0,-1.239504,0.196178,-0.940457,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


In [279]:
cat_fet = X.columns

fet_idx = np.arange(len(cat_fet))
fet_idx = np.delete( fet_idx, [4, 17, 18])

In [280]:
fet_idx

array([ 0,  1,  2,  3,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 19,
       20, 21, 22, 23, 24, 25, 26, 27, 28, 29])

In [281]:
from imblearn.over_sampling import SMOTENC

nc = SMOTENC(categorical_features = fet_idx.tolist(), random_state= 42)

X_nc, y_nc = nc.fit_resample(X_train, y_train)

In [282]:
model_nc1 = LogisticRegression(
    random_state=42,
    class_weight='balanced',
)

model_nc1.fit(X_nc, y_nc)
pred_nc1 = model_nc1.predict_proba(X_test)[:, 1]
pred_nc1 = (np.array(pred_nc1) >= 0.5).astype(int)

f1_score(y_test, pred_nc1)

0.6048565121412803

In [283]:
model_nc = RandomForestClassifier(
    random_state=42,
    class_weight='balanced',
    max_depth=8,
    min_samples_split=9,
    n_estimators=100,
)

model_nc.fit(X_nc, y_nc)
pred_nc = model_nc.predict_proba(X_test)[:, 1]
pred_nc = (np.array(pred_nc) >= 0.5).astype(int)

f1_score(y_test, pred_nc)




0.6326530612244898

In [287]:
from catboost import CatBoostClassifier


model_nc = CatBoostClassifier(
    random_seed=42,
    learning_rate=0.01,
    loss_function='Logloss',
    eval_metric='F1',
    iterations=2000,
    depth=4,
    l2_leaf_reg=100,
    )

model_nc.fit(X_nc, y_nc)
pred_nc = model_nc.predict_proba(X_test)[:, 1]
pred_nc = (np.array(pred_nc) >= 0.5).astype(int)

f1_score(y_test, pred_nc)

0:	learn: 0.8036957	total: 2.92ms	remaining: 5.83s
1:	learn: 0.7949402	total: 6.04ms	remaining: 6.03s
2:	learn: 0.7916416	total: 8.66ms	remaining: 5.76s
3:	learn: 0.8018766	total: 11.3ms	remaining: 5.62s
4:	learn: 0.8094331	total: 14.5ms	remaining: 5.79s
5:	learn: 0.8101970	total: 17.3ms	remaining: 5.74s
6:	learn: 0.8129529	total: 19.8ms	remaining: 5.64s
7:	learn: 0.8136590	total: 22.4ms	remaining: 5.57s
8:	learn: 0.8107370	total: 24.9ms	remaining: 5.51s
9:	learn: 0.8110326	total: 28.6ms	remaining: 5.7s
10:	learn: 0.8122500	total: 31.3ms	remaining: 5.66s
11:	learn: 0.8112210	total: 33.9ms	remaining: 5.62s
12:	learn: 0.8123499	total: 36.5ms	remaining: 5.58s
13:	learn: 0.8127217	total: 39.1ms	remaining: 5.54s
14:	learn: 0.8134816	total: 41.6ms	remaining: 5.51s
15:	learn: 0.8116571	total: 45.7ms	remaining: 5.66s
16:	learn: 0.8126861	total: 48.6ms	remaining: 5.67s
17:	learn: 0.8126861	total: 51.4ms	remaining: 5.66s
18:	learn: 0.8124070	total: 54ms	remaining: 5.63s
19:	learn: 0.8130305	tota

0.6298472385428907

In [288]:
import xgboost as xgb

model = xgb.XGBClassifier(
    n_estimators=50,
    random_state=42
)
model.fit(X_train, y_train)

# Предсказание и оценка точности
y_pred = model.predict(X_test)
f1_score(y_test, y_pred)

0.5750708215297451